In [1]:
"""
Test TypeInferenceEngine.get_type() method
Copy and paste this into a Jupyter notebook cell
"""

import ast
from analyzer.analysis.expression_traversal import TypeInferenceEngine
from analyzer.analysis import Scope

# Create a mock project node (not used for literals/simple lookups yet)
class MockProjectNode:
    pass

project = MockProjectNode()
engine = TypeInferenceEngine(project)

# Create a scope with some test variables
scope = Scope()
scope.push_frame()
scope.add("x", "int")
scope.add("name", "str")
scope.add("is_valid", "bool")

print("=" * 70)
print("TypeInferenceEngine.get_type() Test Results")
print("=" * 70)

# Test cases
test_cases = [
    # Literals
    ("42", "Literal integer"),
    ("3.14", "Literal float"),
    ('"hello"', "Literal string"),
    ("True", "Literal boolean"),
    ("False", "Literal boolean"),
    ("None", "Literal None"),
    
    # Variable lookups (from scope)
    ("x", "Variable lookup (int)"),
    ("name", "Variable lookup (str)"),
    ("is_valid", "Variable lookup (bool)"),
    ("unknown_var", "Unknown variable (not in scope)"),
]

for expr_string, description in test_cases:
    print(f"\n{description}")
    print(f"Expression: {expr_string}")
    print("-" * 70)
    
    # Parse the expression
    expr_ast = ast.parse(expr_string, mode='eval').body
    
    # Get the type
    type_fqn = engine.get_type(expr_ast, scope)
    
    # Display result
    if type_fqn:
        print(f"✓ Type: {type_fqn}")
    else:
        print(f"✗ Type: None (could not determine)")

print("\n" + "=" * 70)
print("Test Complete!")
print("=" * 70)

TypeInferenceEngine.get_type() Test Results

Literal integer
Expression: 42
----------------------------------------------------------------------
✓ Type: int

Literal float
Expression: 3.14
----------------------------------------------------------------------
✓ Type: float

Literal string
Expression: "hello"
----------------------------------------------------------------------
✓ Type: str

Literal boolean
Expression: True
----------------------------------------------------------------------
✓ Type: bool

Literal boolean
Expression: False
----------------------------------------------------------------------
✓ Type: bool

Literal None
Expression: None
----------------------------------------------------------------------
✓ Type: NoneType

Variable lookup (int)
Expression: x
----------------------------------------------------------------------
✓ Type: int

Variable lookup (str)
Expression: name
----------------------------------------------------------------------
✓ Type: str

Varia

In [2]:
"""
Test ModuleAnalysisVisitor with Type Inference
Copy and paste this into a Jupyter notebook cell
"""

import ast
from analyzer.analysis.visitors.module_analysis_visitor import ModuleAnalysisVisitor

# Create a simple test module with various assignments
test_code = """
# Literal assignments
x = 42
name = "Alice"
pi = 3.14
is_valid = True
result = None

# Variable-to-variable assignment (should lookup in scope)
y = x
greeting = name

# Annotated assignment
count: int = 100
"""

# Parse the test code
module_ast = ast.parse(test_code)

# Create a mock module node
class MockModuleNode:
    def __init__(self, ast_module):
        self.source_data = type('obj', (object,), {'ast_node': ast_module})()
        self.name = "test_module"
    
    def get_project(self):
        # Return mock project node
        class MockProjectNode:
            pass
        return MockProjectNode()

# Create the module node
module_node = MockModuleNode(module_ast)

# Create and run the visitor
print("=" * 70)
print("ModuleAnalysisVisitor Type Inference Test")
print("=" * 70)
print()

visitor = ModuleAnalysisVisitor(module_node)
visitor.visit(module_ast)

print()
print("=" * 70)
print("Final Scope Contents:")
print("=" * 70)

# Display what's in the scope
if visitor.scope._frames:
    frame = visitor.scope._frames[0]
    if frame._bindings:
        for var_name, type_fqn in frame._bindings.items():
            print(f"  {var_name}: {type_fqn}")
    else:
        print("  (empty)")
else:
    print("  (no frames)")

print()
print(f"Total assignments processed: {visitor.assignment_count}")
print()
print("=" * 70)
print("Test Complete!")
print("=" * 70)

ModuleAnalysisVisitor Type Inference Test

   Inferred: x = int (line 3)
   Inferred: name = str (line 4)
   Inferred: pi = float (line 5)
   Inferred: is_valid = bool (line 6)
   Inferred: result = NoneType (line 7)
   Inferred: y = int (line 10)
   Inferred: greeting = str (line 11)
   Inferred: count: ... = int (line 14)

Final Scope Contents:
  x: int
  name: str
  pi: float
  is_valid: bool
  result: NoneType
  y: int
  greeting: str
  count: int

Total assignments processed: 8

Test Complete!
